In [1]:
# ==========================================
# CELL 1 — Imports & Config
# ==========================================
import pandas as pd
import os
import logging
from datetime import datetime

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)

# Bronze source (the CSV you saved earlier)
BRONZE_CSV_PATH = "/lakehouse/default/Files/bronze/bronze_bursa_banks.csv"

# Silver output location (Files section, for CSV backup)
SILVER_FILES_PATH = "/lakehouse/default/Files/silver"
os.makedirs(SILVER_FILES_PATH, exist_ok=True)

# Mapping dictionary for Ticker to Readable Stock Name
TICKER_TO_NAME_MAP = {
    "1155.KL": "MAYBANK",
    "1295.KL": "PUBLIC BANK",
    "1023.KL": "CIMB",
    "1066.KL": "RHB BANK",
    "5258.KL": "BIMB"
}


# ==========================================
# CELL 2 — Load Bronze data
#           (replaces load_bronze_data() which scanned local parquet files)
# ==========================================
def load_bronze_data(csv_path: str) -> pd.DataFrame:
    """Reads the Bronze CSV from the Lakehouse Files section."""
    logging.info(f"Loading raw data from Bronze layer: {csv_path}")

    if not os.path.exists(csv_path):
        logging.error(f"Bronze file not found at {csv_path}")
        return pd.DataFrame()

    df = pd.read_csv(csv_path)
    logging.info(f"Successfully loaded {len(df)} total rows from Bronze.")
    return df


# ==========================================
# CELL 3 — Clean & conform (unchanged logic from your original script)
# ==========================================
def clean_and_conform(df: pd.DataFrame) -> pd.DataFrame:
    """
    Applies data quality rules: deduplication, type casting, name mapping,
    and keeping ONLY valid trading days.
    """
    logging.info("Starting data cleansing and conformation...")

    # 1. Ensure correct data types
    df['date'] = pd.to_datetime(df['date']).dt.date
    numeric_cols = ['open', 'high', 'low', 'close', 'volume']
    for col in numeric_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')

    # 2. Deduplicate: keep the latest ingestion for each ticker + date combination
    df = df.sort_values(by=['ticker_symbol', 'date', 'ingestion_timestamp'], ascending=[True, True, False])
    initial_row_count = len(df)
    df = df.drop_duplicates(subset=['ticker_symbol', 'date'], keep='first')
    dropped_rows = initial_row_count - len(df)
    if dropped_rows > 0:
        logging.info(f"Dropped {dropped_rows} duplicate rows from multiple runs.")

    # 3. Add readable stock name column
    logging.info("Mapping ticker symbols to readable stock names...")
    df['stock_name'] = df['ticker_symbol'].map(TICKER_TO_NAME_MAP)
    df['stock_name'] = df['stock_name'].fillna(df['ticker_symbol'])

    # 4. Remove any rows with missing core price data
    logging.info("Dropping any rows with missing price data...")
    initial_clean_count = len(df)
    df = df.dropna(subset=['date', 'close'])
    dropped_na = initial_clean_count - len(df)
    if dropped_na > 0:
        logging.info(f"Dropped {dropped_na} rows with missing 'close' prices.")

    # Ensure volume is clean
    df['volume'] = df['volume'].fillna(0).astype(int)

    # 5. Sort chronologically
    df = df.sort_values(by=['stock_name', 'date']).reset_index(drop=True)

    # 6. Add derived Silver-level metadata
    df['silver_processed_at'] = datetime.now()

    logging.info("Data cleansing complete.")
    return df


# ==========================================
# CELL 4 — Run extraction + cleaning
# ==========================================
raw_df = load_bronze_data(BRONZE_CSV_PATH)

if raw_df.empty:
    raise ValueError("No Bronze data loaded — check the Bronze CSV path/step before continuing.")

silver_df = clean_and_conform(raw_df)
silver_df.head()


# ==========================================
# CELL 5 — Save to Silver: managed Delta table (for Gold + Power BI)
# ==========================================
# date columns need to be proper datetime types for Spark/Delta
silver_df_for_spark = silver_df.copy()
silver_df_for_spark['date'] = pd.to_datetime(silver_df_for_spark['date'])
silver_df_for_spark['ingestion_timestamp'] = pd.to_datetime(silver_df_for_spark['ingestion_timestamp'])
silver_df_for_spark['silver_processed_at'] = pd.to_datetime(silver_df_for_spark['silver_processed_at'])

spark_df = spark.createDataFrame(silver_df_for_spark)
spark_df.write.format("delta").mode("overwrite").saveAsTable("silver_bursa_banks")

logging.info(f"✅ Saved {silver_df.shape[0]} rows -> Lakehouse table: silver_bursa_banks")


# ==========================================
# CELL 6 — ALSO save a CSV copy into Files (for manual inspection, matches your original workflow)
# ==========================================
csv_path = f"{SILVER_FILES_PATH}/silver_bursa_banks.csv"
silver_df.to_csv(csv_path, index=False)
logging.info(f"✅ Saved CSV -> {csv_path}")


# ==========================================
# CELL 7 — Quick verification
# ==========================================
spark.sql("SELECT stock_name, COUNT(*) as row_count FROM silver_bursa_banks GROUP BY stock_name").show

StatementMeta(, 407933ab-4209-4c82-96c4-8712571bc293, 3, Finished, Available, Finished, False)

2026-09-04 03:35:21,296 - INFO - Loading raw data from Bronze layer: /lakehouse/default/Files/bronze/bronze_bursa_banks.csv
2026-09-04 03:35:21,710 - INFO - Successfully loaded 3295 total rows from Bronze.
2026-09-04 03:35:21,711 - INFO - Starting data cleansing and conformation...
2026-09-04 03:35:21,780 - INFO - Mapping ticker symbols to readable stock names...
2026-09-04 03:35:21,783 - INFO - Dropping any rows with missing price data...
2026-09-04 03:35:21,789 - INFO - Data cleansing complete.
2026-09-04 03:35:39,902 - INFO - ✅ Saved 3295 rows -> Lakehouse table: silver_bursa_banks
2026-09-04 03:35:40,065 - INFO - ✅ Saved CSV -> /lakehouse/default/Files/silver/silver_bursa_banks.csv


<bound method DataFrame.show of DataFrame[stock_name: string, row_count: bigint]>